# Simple data center: constant-load sizing with DC economics
Keep wind, PV and battery capacities fixed. One integrated CPLEX model chooses the
constant DC load and hourly dispatch under the YAML reliability target.

Objective: served IT revenue minus annualized DC cost and optional fluctuation penalty [EUR].
Facility and IT CAPEX use separate CRFs; fixed OPEX scales with installed MW_IT.
All economic coefficients come from the shared simple_datacenter.yml.
No PPA payments or HPP costs/revenue enter the economic sizing objective.

Select hydesign_clean and run all cells. No other notebook needs to be run first.

The EMS uses one CPLEX dispatch solve. Set battery_power_fluctuation_penalty
in YAML (default 0.0; optional trials 0.0001 or 0.001).
Its units are MWh_IT per MW of adjacent change in AC net battery power
(discharge - charge); there is no end-to-start ramp or throughput penalty.
Fixed/basic dispatch maximizes served IT MWh minus this energy-equivalent penalty.
Economic sizing multiplies the penalty by the IT revenue rate to obtain EUR.
Economic reports exclude the artificial penalty; solver objectives/bounds include it.

The unchanged validator rejects simultaneous charging and discharging. The existing
physical formulation permits this, and neither zero nor a positive ramp penalty
guarantees its elimination. The annual reference evaluation can therefore stop with
"Simultaneous battery charging/discharging"; do not use rejected dispatch as a result.


In [ ]:
import os
os.environ['OPENMDAO_REPORTS'] = '0'
import time
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import hydesign

repo = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'hydesign/ems/ems_simple_datacenter.py').is_file()), None)
if repo is None or Path(hydesign.__file__).resolve().parent != (repo / 'hydesign').resolve():
    raise RuntimeError('Select the kernel with this HyDesign checkout installed editable.')
print('Python:', sys.executable)
print('HyDesign:', hydesign.__file__)

from hydesign.assembly.hpp_assembly_simple_datacenter import hpp_model_simple_datacenter as hpp_model
from hydesign.examples import examples_filepath
from hydesign.sizing_simple_datacenter_with_dc_cost import size_datacenter_with_dc_cost as size_datacenter


## Site and sizing assumptions
The minimum reliability, load bounds and solver limits come from the shared YAML.
The 16 MW value below is used only by the existing assembly during input preparation.


In [ ]:
examples_sites = pd.read_csv(f'{examples_filepath}examples_sites.csv', index_col=0, sep=';')
name = 'Denmark_good_solar'
ex_site = examples_sites.loc[examples_sites.name == name].iloc[0]
latitude = ex_site.latitude
longitude = ex_site.longitude
altitude = ex_site.altitude
input_ts_fn = examples_filepath + ex_site.input_ts_fn
sim_pars_fn = examples_filepath + 'thesis_simple_datacenter/simple_datacenter.yml'

facility_load_mw = 16.0
PUE = 1.15
PPA = 70.0 # Reporting only; unused by dispatch/sizing objectives.
hpp = hpp_model(
    latitude=latitude,
    longitude=longitude,
    altitude=altitude,
    work_dir='./',
    sim_pars_fn=sim_pars_fn,
    input_ts_fn=input_ts_fn,
    facility_load_mw=facility_load_mw,
    pue=PUE,
    ppa_price=PPA,
)
reliability_target = hpp.sim_pars['reliability_target']
print(f'Minimum full-load factor for sizing: {reliability_target:.2%}')
print('Site:', name)
print('Weather:', input_ts_fn)
print('Preparation load [MW] (not the sized result):', facility_load_mw)
print('Hourly weather:', hpp.weather.index[0], 'to', hpp.weather.index[-1], 'N_time =', hpp.N_time)


## Fixed HPP and generation inputs
Wind is 100 MW, PV is 180 MW AC, and the battery is 25 MW / 200 MWh nominal.
These capacities remain fixed during load sizing.

The existing hpp.evaluate(*x) call prepares hourly wind and PV generation. It also
runs an unconstrained fixed-load dispatch because that is part of the existing assembly.
That dispatch is not the sizing result, and its battery schedule is not passed to sizing.
Only the generation profiles and fixed plant parameters are reused; the next solve
chooses its own load and dispatch with the YAML reliability target enforced.


In [ ]:
start = time.time()
clearance = 35
sp = 300
p_rated = 5
Nwt = 20
wind_MW_per_km2 = 7
solar_MW = 180
surface_tilt = 42
surface_azimuth = 180
solar_DCAC = 1.25
b_P = 25
b_E_h = 8

x = [clearance, sp, p_rated, Nwt, wind_MW_per_km2,
     solar_MW, surface_tilt, surface_azimuth, solar_DCAC,
     b_P, b_E_h]
# Prepare generation using the existing fixed-design assembly.
# Disable reliability only for this preparation; keep the YAML target for sizing.
hpp.prob.set_val('reliability_target', 0.0)
hpp.evaluate(*x)
print('Generation preparation time [min]:', (time.time() - start) / 60)
print('Hourly generation prepared:', len(hpp.prob.get_val('wind_t')))


## Joint DC sizing and dispatch with economics
The original assembly above prepares wind/PV profiles and runs its fixed-load
preparation dispatch. Only generation profiles and fixed plant parameters are reused.
The sizing solve creates its own load and dispatch; no preparation battery schedule is used.

Revenue = YAML firm IT value * actually served IT energy. Annualized cost comes
from the standalone cost model evaluated per MW_IT, multiplied by installed IT
capacity (= facility_load / PUE). Reliability and physical constraints are unchanged.

One solve maximizes net annual DC value minus the optional fluctuation penalty in EUR.
A time-limit feasible result is accepted only when enabled in YAML and is labelled
NOT proven optimal, with its remaining net-value gap and bound.


In [ ]:
sim_pars = hpp.sim_pars
result = size_datacenter(
    wind_ts=hpp.prob.get_val('wind_t'),
    solar_ts=hpp.prob.get_val('solar_t'),
    b_P=b_P,
    b_E=b_P * b_E_h,
    battery_depth_of_discharge=sim_pars['battery_depth_of_discharge'],
    battery_charge_efficiency=sim_pars['battery_charge_efficiency'],
    battery_initial_soc=sim_pars['battery_initial_soc'],
    pue=PUE,
    reliability_target=reliability_target,
    battery_power_fluctuation_penalty=sim_pars.get('battery_power_fluctuation_penalty', 0.0),
    datacenter_facility_capex_per_mw_it=sim_pars['datacenter_facility_capex_per_mw_it'],
    datacenter_it_capex_per_mw_it=sim_pars['datacenter_it_capex_per_mw_it'],
    datacenter_fixed_opex_per_mw_it_year=sim_pars['datacenter_fixed_opex_per_mw_it_year'],
    datacenter_discount_rate=sim_pars['datacenter_discount_rate'],
    datacenter_facility_lifetime_years=sim_pars['datacenter_facility_lifetime_years'],
    datacenter_it_lifetime_years=sim_pars['datacenter_it_lifetime_years'],
    datacenter_firm_revenue_per_mwh_it=sim_pars['datacenter_firm_revenue_per_mwh_it'],
    facility_load_min_mw=sim_pars['facility_load_min_mw'],
    facility_load_max_mw=sim_pars['facility_load_max_mw'],
    epsilon_mw=sim_pars['epsilon_mw'],
    time_limit_seconds=sim_pars['sizing_time_limit_seconds'],
    threads=sim_pars['ems_threads'],
    mip_relative_gap=sim_pars['sizing_mip_relative_gap'],
    allow_time_limit_incumbent=sim_pars['sizing_allow_time_limit_incumbent'],
)


## Sized load and hourly results
All metrics below describe the load and dispatch selected by the sizing optimization.


In [ ]:
print('Solver:', result['solver_status'])
print('Single-stage status:', result['solver_status']['stages'][0]['status'])
print(f"Primary net annual value: EUR {result['solver_status']['primary_net_annual_value']:,.2f}/year")
if result['solver_status']['optimal']:
    print('Optimal within the configured CPLEX tolerance.')
else:
    print('FEASIBLE CANDIDATE - NOT PROVEN OPTIMAL')
primary = result['solver_status']['stages'][0]
print('Objective units:', primary['objective_units'])
print('Unpenalized physical/economic objective:', primary['unpenalized_objective'])
print('Net battery-power fluctuation [MW]:', primary['battery_power_fluctuation_mw'])
print('Artificial penalty:', primary.get('battery_penalty_eur', primary.get('battery_penalty_equivalent_it_mwh')))
print(f"Remaining penalized DC-objective gap: {primary.get('relative_gap', 0.0):.2%}")
print(f"Penalized DC-objective upper bound: EUR {primary.get('best_bound', primary['objective']):,.2f}/year")
print(f"Contracted facility load: {result['facility_load_mw']:.6f} MW")
print(f"Corresponding IT load: {result['firm_demand_it_mw']:.6f} MW")
print('Load bounds [MW]:', result['facility_load_bounds_mw'])
print('Necessary physical upper bound [MW]:', result['physical_load_upper_mw'])
print('Upper/lower bound active:', result['upper_bound_active'], result['lower_bound_active'])
if result['upper_bound_active']:
    print('WARNING: increase the upper load bound before interpreting this as the unrestricted optimum.')

metrics = result['metrics']
print(f"Full-load factor: {metrics['full_load_factor']:.4%} (minimum {reliability_target:.2%})")
print(f"Failed hours: {metrics['LOLH']:.0f} / {metrics['total_hours']:.0f}")
print(f"Delivered facility energy: {metrics['delivered_energy_mwh']:.3f} MWh/year")
print(f"EENS (reported only): {metrics['EENS']:.3f} MWh/year")
print(f"Longest chronological shortage: {metrics['longest_shortage_hours']:.0f} h")
print(f"Minimum supplied facility power: {metrics['minimum_supplied_mw']:.3f} MW")
print(f"Maximum shortfall: {metrics['maximum_shortfall_mw']:.3f} MW")
allowed_failed_hours = int(np.floor(
    (1 - reliability_target) * metrics['total_hours'] + 1e-9
))
assert metrics['LOLH'] <= allowed_failed_hours
assert metrics['full_load_factor'] >= reliability_target - 1e-12
print('Physical validation passed:', result['validation'])

# The sizing solution is in result; hpp.prob contains only the preparation run.
# All results and plots below use the sizing solution.
hourly = pd.DataFrame(result['hourly'], index=hpp.weather.index)
soc_index = hpp.weather.index.append(pd.DatetimeIndex([hpp.weather.index[-1] + pd.Timedelta(hours=1)]))
soc_mwh = pd.Series(result['soc_mwh'], index=soc_index, name='soc_mwh')
P_supplied = hourly.served_facility_mw.to_numpy()
display(hourly.head())


## Sized data-center economics
All values below refer to the selected DC size and its optimized dispatch.


In [ ]:
economics = result['economics']
print(f"Facility load: {economics['facility_load_mw']:.6f} MW")
print(f"Installed IT capacity: {economics['it_capacity_mw']:.6f} MW_IT")
print(f"Served IT energy: {economics['served_it_energy_mwh']:,.3f} MWh_IT/year")
print(f"Served facility energy: {economics['served_facility_energy_mwh']:,.3f} MWh/year")
print(f"Firm IT value: {economics['datacenter_firm_revenue_per_mwh_it']:,.2f} EUR/MWh_IT")
for name in ('CAPEX_dc_facility', 'CAPEX_dc_it', 'CAPEX_dc_total'):
    print(f"{name}: EUR {economics[name]:,.2f}")
for name in ('Revenue_dc', 'OPEX_dc', 'annualized_CAPEX_dc_facility',
             'annualized_CAPEX_dc_it', 'annualized_CAPEX_dc_total',
             'annualized_cost_dc', 'net_annual_value'):
    print(f"{name}: EUR {economics[name]:,.2f}/year")
print(f"Curtailment: {metrics['curtailment_mwh']:,.3f} MWh/year")
print(f"Battery throughput (charge + discharge): {metrics['battery_throughput_mwh']:,.3f} MWh/year")


## Sized-load dispatch plots
These plots use result['hourly'] and result['soc_mwh'] from the sizing solve.


In [ ]:
# Plot a week around the largest shortfall, or the first week if none occurs.
peak = int(np.argmax(hourly.unserved_facility_mw.to_numpy()))
start = max(0, min(peak - 72, len(hourly) - 168))
stop = min(start + 168, len(hourly))
window = hourly.iloc[start:stop]
fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
axes[0].plot(window.index, window.renewable_available_mw, label='Available wind + PV')
axes[0].plot(window.index, window.served_facility_mw, label='Served facility load')
axes[0].plot(window.index, window.facility_load_mw, '--', label='Requested facility load')
axes[0].plot(window.index, window.unserved_facility_mw, label='Unserved facility load')
axes[0].set_ylabel('MW'); axes[0].legend()
axes[1].plot(window.index, window.discharge_mw - window.charge_mw, label='Battery: + discharge / - charge')
axes[1].axhline(0, color='grey', linewidth=0.5)
axes[1].set_ylabel('MW'); axes[1].legend()
axes[2].plot(soc_mwh.iloc[start:stop+1], label='Stored energy')
axes[2].axhline((1-hpp.sim_pars['battery_depth_of_discharge'])*b_P*b_E_h, linestyle='--', color='grey')
axes[2].axhline(b_P*b_E_h, linestyle='--', color='grey')
axes[2].set_ylabel('MWh'); axes[2].legend()
fig.autofmt_xdate(); plt.tight_layout(); plt.show()


## Formulation and limits
The sizing variable is facility_load; installed it_capacity_mw = facility_load / PUE.
The unpenalized economic value is Revenue_dc - annualized_cost_dc. All served IT energy,
including partial supply in failed hours, earns the same YAML revenue rate.

The existing hourly EMS constraint builder is reused without modifications: battery
power convention, efficiency, SOC bounds and boundaries, PUE, curtailment, Big-M,
failed-hour budget and the physical sizing upper bound remain unchanged.

There is no secondary solve or objective lock. The selected load and failure
permissions remain optimization variables throughout the single dispatch solve.

This is one 8760-hour weather year with perfect foresight. No HPP economics, PPA
payments, workload flexibility, degradation or lifetime cash-flow model is included.
The 5-year IT lifetime represents replacement through its CRF only.
